# OSM: puntos de encuentro

Análisis exploratorio independiente de bronce. Fecha de revisión: 13 de noviembre de 2026.

Se examinan los archivos locales originales, no una tabla Delta ejecutada en Databricks.
No se limpia ni sobrescribe la fuente. Las transformaciones que aparezcan se hacen en copias
para evaluar lo que necesitará plata. Los resultados no prueban causalidad ni certifican seguridad.

Cada notebook sigue el orden: procedencia y unidad, calidad por columna y geometría,
distribuciones, límites y reglas para plata. No se muestran registros individuales de edificios oficiales.

In [ ]:
from pathlib import Path
import sys, json, re, unicodedata
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "CLAUDE.md").exists() and (p / "data/externos").is_dir())
sys.path.insert(0, str(ROOT / "analisis_bronce"))
from soporte import (ficha, huella, leer_geojson, mostrar_perfil, geometria, conteos,
                     faltantes, guardar_resumen, grafico_barras, SALIDAS, CRS_METROS)
pd.set_option("display.max_columns", 24)
pd.set_option("display.max_rows", 60)
plt.rcParams.update({"font.size": 11, "figure.dpi": 110})

## 1. Procedencia y unidad de análisis

Archivo: `data/externos/osm_puntos_encuentro.geojson`. Una fila representa un punto de encuentro etiquetado como emergency=assembly_point, cuando exista.
La fecha de descarga no equivale a fecha de verificación del objeto en terreno.

In [ ]:
PATH = ROOT / 'data/externos/osm_puntos_encuentro.geojson'
meta = ficha(PATH)
raw, df, g = leer_geojson(PATH)
print("Archivo:", meta["archivo"], "| filas:", len(df), "| propiedades:", len(df.columns))
display(pd.Series(meta, name="procedencia").to_frame())

## 2. Calidad por columna y geometría

In [ ]:
calidad = mostrar_perfil(df)
geo = geometria(g)
display(pd.Series(geo, name="geometria").to_frame())
resumen = {"dataset": '07_osm_puntos_encuentro', **meta, "filas": len(df), "columnas": list(df.columns),
           "calidad_columnas": calidad.to_dict("records"), "geometria": geo}

## 3. Vacío de cobertura y contrato de una capa opcional

In [ ]:
print("Tipo de contenedor:", raw.get("type"))
print("Número de features:",len(raw.get("features",[])))
print("Llaves del contenedor:",list(raw))
resumen["capa_vacia"] = len(df)==0
resumen["interpretacion"] = "Sin puntos mapeados en el extracto de Cali; no prueba inexistencia de puntos reales"
display(pd.DataFrame([{"comprobacion":"GeoJSON interpretable","resultado":raw.get("type")=="FeatureCollection"},
                      {"comprobacion":"Hay observaciones para analizar","resultado":len(df)>0}]))

## Reglas decididas para plata

- **No usar como inventario de puntos habilitados.** Conservar como evidencia de cobertura OSM vacía.
- Crear una tabla plata opcional vacía con esquema explícito si el sistema la necesita, conservando `estado_fuente=sin_registros`.
- No interpretar el vacío como cero necesidad ni imputar puntos a partir de colegios.
- Buscar un inventario de la autoridad local para habilitar este uso.

In [ ]:
assert huella(PATH) == meta["sha256"], "La fuente cruda cambió durante el análisis"
guardar_resumen('07_osm_puntos_encuentro', resumen)
print("Resumen y gráficos guardados; la fuente cruda conserva su SHA-256.")